# Cruzamento — Candidatos + Resultados
Fase: obter o `Status` real por (candidato, curso), a unidade correta de análise.

**Decisão de modelação confirmada:** a admissão é decidida por curso, não por candidato de forma
genérica — o mesmo candidato pode ser admitido numa opção e não admitido noutra. Por isso o
ficheiro de candidatos (formato largo: `UEM_Cod_Opc1`/`UEM_Cod_Opc2` em colunas) precisa de ser
transformado para formato longo (uma linha por candidato+curso) antes do cruzamento.

**`Obs = 'A'` significa Ausente** — candidato não compareceu ao exame. Isto é tratado como
categoria própria, nunca como "reprovado", para não confundir "não sabia a matéria" com
"não apareceu".

In [3]:
import pandas as pd

pd.set_option('display.max_columns', None)

df_candidatos = pd.read_parquet('candidatos_uem.parquet')
df_resultados = pd.read_excel('UEMResultadosFinal-APURAMENTO2026.xlsx') 

print("Candidatos:", df_candidatos.shape)
print("Resultados:", df_resultados.shape)
df_resultados.head()

Candidatos: (26798, 23)
Resultados: (51129, 16)


,Prov,NoCand,Nome,CursoID,Curso,Discip1,Nota1,Discip2,Nota2,Media,Discip3,Nota3,Obs,Resultados,Sexo,data_Nasc
0,Gaza,10019,SONIA SUARES,10100,Admin. Pública - Diurno - UEM,História-I,0.00,Português-I,0.0,0.00,NaN,NaN,A,Não admitido,F,2011-12-11
1,Gaza,10019,SONIA SUARES,10128,Arqueologia e Gest. Patr. Cultural - Diurno - UEM,Geografia-I,0.00,História-II,0.0,0.00,NaN,NaN,A,Não admitido,F,2011-12-11
2,Cidade de Maputo,10036,ELISABETH OLGA MASSANGO,11100,Engenharia Civil - Diurno - UEM,Física-I,5.40,Matemática-I,4.3,4.85,NaN,NaN,NaN,Não admitido,F,2006-04-07
3,Cidade de Maputo,10036,ELISABETH OLGA MASSANGO,11108,Engenharia Química - Diurno - UEM,Física-I,5.40,Matemática-I,4.3,4.85,NaN,NaN,NaN,Não admitido,F,2006-04-07
4,Cidade de Maputo,10037,HAWA NATURALAMA ABOO CHIRIDA,11108,Engenharia Química - Diurno - UEM,Física-I,7.84,Matemática-I,4.7,6.27,NaN,NaN,NaN,Não admitido,F,2004-12-06


## 1. Limpeza inicial do ficheiro de resultados

In [4]:
df_resultados.info()
print()
print("Valores únicos em Resultados:", df_resultados['Resultados'].unique())
print("Valores únicos em Obs:", df_resultados['Obs'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 51129 entries, 0 to 51128
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Prov        51129 non-null  str           
 1   NoCand      51129 non-null  int64         
 2   Nome        51129 non-null  str           
 3   CursoID     51129 non-null  int64         
 4   Curso       51129 non-null  str           
 5   Discip1     51129 non-null  str           
 6   Nota1       51129 non-null  float64       
 7   Discip2     51129 non-null  str           
 8   Nota2       51129 non-null  float64       
 9   Media       51129 non-null  float64       
 10  Discip3     121 non-null    str           
 11  Nota3       121 non-null    str           
 12  Obs         2459 non-null   str           
 13  Resultados  51129 non-null  str           
 14  Sexo        50966 non-null  str           
 15  data_Nasc   50966 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(3),

In [ ]:
# padronizar chave para bater com o tipo do ficheiro de candidatos
df_resultados['NoCand'] = df_resultados['NoCand'].astype(str).str.strip()
df_resultados['CursoID'] = df_resultados['CursoID'].astype(int)

# padronizar texto
df_resultados['Resultados'] = df_resultados['Resultados'].str.strip().str.title()
df_resultados['Obs'] = df_resultados['Obs'].fillna('').str.strip()

# criar coluna de status final considerando ausência
df_resultados['status_final'] = df_resultados.apply(
    lambda row: 'Ausente' if row['Obs'] == 'A' else row['Resultados'],
    axis=1
)

df_resultados['status_final'].value_counts()

## 2. Transformar candidatos de formato largo para longo
Cada candidato tem até 2 opções de curso (`UEM_Cod_Opc1`, `UEM_Cod_Opc2`). Para cruzar com os
resultados (uma linha por candidato+curso), precisamos de uma linha por opção.

In [ ]:
df_candidatos['candidato_codigo'] = df_candidatos['candidato_codigo'].astype(str).str.strip()

opc1 = df_candidatos.copy()
opc1['curso_codigo'] = opc1['UEM_Cod_Opc1']
opc1['curso_nome'] = opc1['UEM_Opc1']
opc1['prioridade'] = 1

opc2 = df_candidatos[df_candidatos['UEM_Cod_Opc2'] != 0].copy()
opc2['curso_codigo'] = opc2['UEM_Cod_Opc2']
opc2['curso_nome'] = opc2['UEM_Opc2']
opc2['prioridade'] = 2

colunas_base = [c for c in df_candidatos.columns if c not in
                ['UEM_Cod_Opc1', 'UEM_Opc1', 'UEM_Cod_Opc2', 'UEM_Opc2']]

df_candidatos_longo = pd.concat([
    opc1[colunas_base + ['curso_codigo', 'curso_nome', 'prioridade']],
    opc2[colunas_base + ['curso_codigo', 'curso_nome', 'prioridade']]
], ignore_index=True)

print("Candidatos formato longo:", df_candidatos_longo.shape)
df_candidatos_longo.head()

## 3. Cruzamento pela chave composta (candidato + curso)

In [ ]:
df_final = df_candidatos_longo.merge(
    df_resultados[['NoCand', 'CursoID', 'Nota1', 'Nota2', 'Media', 'Nota3', 'status_final']],
    left_on=['candidato_codigo', 'curso_codigo'],
    right_on=['NoCand', 'CursoID'],
    how='left'
)

print("Resultado do merge:", df_final.shape)
df_final[['candidato_codigo', 'curso_codigo', 'prioridade', 'status_final']].head(10)

## 4. Verificar qualidade do cruzamento
Antes de seguir, precisamos de saber: quantas linhas de candidatos ficaram sem resultado
correspondente? Isto revela problemas de chave (formato diferente, candidato sem resultado
registado ainda, etc.) — não avançar sem entender esta percentagem.

In [ ]:
sem_resultado = df_final['status_final'].isna().sum()
pct_sem_resultado = sem_resultado / len(df_final) * 100
print(f"Linhas candidato+curso sem resultado correspondente: {sem_resultado} ({pct_sem_resultado:.1f}%)")

# se a percentagem for alta, investigar exemplos concretos
if sem_resultado > 0:
    print()
    print("Exemplos sem correspondência:")
    print(df_final[df_final['status_final'].isna()][['candidato_codigo', 'curso_codigo', 'curso_nome']].head(10))

In [ ]:
# checagem inversa: resultados que não bateram com nenhum candidato
chave_candidatos = set(zip(df_candidatos_longo['candidato_codigo'], df_candidatos_longo['curso_codigo']))
chave_resultados = set(zip(df_resultados['NoCand'], df_resultados['CursoID']))

resultados_sem_candidato = chave_resultados - chave_candidatos
print(f"Resultados sem candidato correspondente: {len(resultados_sem_candidato)}")
if resultados_sem_candidato:
    print("Exemplos:", list(resultados_sem_candidato)[:10])

## 5. Distribuição final do target
Com `status_final` definido (Admitido / Não Admitido / Ausente / sem correspondência),
esta é a base real para o modelo — mas as linhas `Ausente` e sem correspondência precisam de
decisão explícita antes do treino, não devem entrar misturadas com Admitido/Não Admitido.

In [ ]:
df_final['status_final'].value_counts(dropna=False)

## 6. Preparar o dataset de treino (separando ausentes)
Regra: o modelo de previsão de admissão só deve aprender com quem realmente fez a prova.
Ausentes e sem correspondência ficam guardados à parte — úteis para métricas de dashboard
(taxa de absentismo), não para o treino do classificador.

In [ ]:
df_treino_ml = df_final[df_final['status_final'].isin(['Admitido', 'Não Admitido'])].copy()
df_ausentes = df_final[df_final['status_final'] == 'Ausente'].copy()
df_sem_correspondencia = df_final[df_final['status_final'].isna()].copy()

print("Base para treino do modelo:", df_treino_ml.shape)
print("Ausentes (guardados à parte):", df_ausentes.shape)
print("Sem correspondência (investigar antes de descartar):", df_sem_correspondencia.shape)

## 7. Guardar os resultados do cruzamento

In [ ]:
df_final.to_parquet('candidatos_resultados_completo.parquet', index=False)
df_treino_ml.to_parquet('base_treino_ml.parquet', index=False)
df_ausentes.to_parquet('candidatos_ausentes.parquet', index=False)

print("Ficheiros guardados.")

## 8. Taxa de admissão — primeira métrica real de negócio
Agora sim, com o target real, dá para responder à pergunta central do produto.

In [ ]:
taxa_admissao_geral = (df_treino_ml['status_final'] == 'Admitido').mean() * 100
print(f"Taxa de admissão geral (excluindo ausentes): {taxa_admissao_geral:.1f}%")

print()
print("Taxa de admissão por curso (top 15 mais procurados):")
taxa_por_curso = df_treino_ml.groupby('curso_nome')['status_final'].apply(
    lambda x: (x == 'Admitido').mean() * 100
).round(1).sort_values(ascending=False)
print(taxa_por_curso.head(15))